In [2]:
import pandas as pd
import random

items = [
    "Milk", "Bread", "Butter", "Eggs", "Cheese",
    "Chips", "Cold Drink", "Biscuits", "Maggi", "Juice"
]

data = []

for i in range(1, 201):  # 200 transactions
    basket = random.sample(items, random.randint(2, 5))
    for item in basket:
        data.append([i, item])

df = pd.DataFrame(data, columns=["Transaction_ID", "Item"])

df.head()


,Transaction_ID,Item
0,1,Butter
1,1,Cold Drink
2,1,Milk
3,1,Maggi
4,1,Biscuits


In [3]:
df.to_csv("grocery_data.csv", index=False)

In [3]:
print("Total rows:", df.shape)
print("Unique transactions:", df['Transaction_ID'].nunique())
print("Unique items:", df['Item'].nunique())

Total rows: (707, 2)
Unique transactions: 200
Unique items: 10


# 🗄️ SQL Integration

In [4]:
import sqlite3
import pandas as pd

In [5]:
df = pd.read_csv("grocery_data.csv")
df.head()

,Transaction_ID,Item
0,1,Cheese
1,1,Cold Drink
2,1,Chips
3,1,Eggs
4,2,Milk


In [6]:
conn = sqlite3.connect("grocery.db")

In [7]:
df.to_sql("transactions", conn, if_exists="replace", index=False)

707

In [8]:
query = """
SELECT Item, COUNT(*) as Frequency
FROM transactions
GROUP BY Item
ORDER BY Frequency DESC
"""

result = pd.read_sql(query, conn)
result

,Item,Frequency
0,Biscuits,81
1,Chips,76
2,Eggs,75
3,Cheese,75
4,Cold Drink,74
5,Butter,74
6,Juice,67
7,Bread,64
8,Maggi,61
9,Milk,60


In [9]:
query = """
SELECT COUNT(DISTINCT Transaction_ID) as Total_Transactions
FROM transactions
"""

pd.read_sql(query, conn)

,Total_Transactions
0,200


In [10]:
conn.close()

In [23]:
import warnings
warnings.filterwarnings("ignore")

In [24]:
!pip install mlxtend

In [25]:
from mlxtend.frequent_patterns import apriori, association_rules

In [26]:
basket = df.groupby(['Transaction_ID', 'Item'])['Item'] \
           .count().unstack().fillna(0)

# Convert counts to 0/1
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

basket.head()

Item,Biscuits,Bread,Butter,Cheese,Chips,Cold Drink,Eggs,Juice,Maggi,Milk
Transaction_ID,,,,,,,,,,
1,0,0,0,1,1,1,1,0,0,0
2,0,0,0,1,0,0,0,0,1,1
3,1,1,0,0,0,0,0,1,0,0
4,1,1,0,0,0,1,1,0,0,0
5,0,0,1,0,0,1,0,0,1,1


In [31]:
frequent_items = apriori(basket, min_support=0.01, use_colnames=True)

frequent_items.head()

,support,itemsets
0,0.405,(Biscuits)
1,0.320,(Bread)
2,0.370,(Butter)
3,0.375,(Cheese)
4,0.380,(Chips)


In [46]:
rules = association_rules(frequent_items, metric="confidence", min_threshold=0.2)

rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Biscuits),(Bread),0.405,0.320,0.125,0.308642,0.964506,1.0,-0.004600,0.983571,-0.058246,0.208333,-0.016703,0.349633
1,(Bread),(Biscuits),0.320,0.405,0.125,0.390625,0.964506,1.0,-0.004600,0.976410,-0.051339,0.208333,-0.024160,0.349633
2,(Butter),(Biscuits),0.370,0.405,0.110,0.297297,0.734067,1.0,-0.039850,0.846731,-0.365094,0.165414,-0.181013,0.284451
3,(Biscuits),(Butter),0.405,0.370,0.110,0.271605,0.734067,1.0,-0.039850,0.864915,-0.378443,0.165414,-0.156183,0.284451
4,(Cheese),(Biscuits),0.375,0.405,0.125,0.333333,0.823045,1.0,-0.026875,0.892500,-0.255952,0.190840,-0.120448,0.320988


In [41]:
rules = rules.sort_values(by='confidence', ascending=False)

rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head()

,antecedents,consequents,support,confidence,lift
747,"(Butter, Cheese, Chips, Milk)",(Bread),0.01,1.0,3.125000
745,"(Butter, Bread, Chips, Milk)",(Cheese),0.01,1.0,2.666667
762,"(Eggs, Cheese, Bread, Milk)",(Butter),0.01,1.0,2.702703
758,"(Eggs, Butter, Bread, Milk)",(Cheese),0.01,1.0,2.666667
574,"(Cheese, Bread, Juice)",(Maggi),0.01,1.0,3.278689


In [48]:
# Step 1: Filter
rules["antecedent_len"] = rules["antecedents"].apply(lambda x: len(x))
filtered_rules = rules[rules["antecedent_len"] <= 2]

# Step 2: Convert to string (for display)
filtered_rules['antecedents'] = filtered_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
filtered_rules['consequents'] = filtered_rules['consequents'].apply(lambda x: ', '.join(list(x)))

## 📊 Key Insights

- Customers who buy Milk often buy Bread  
- Chips and Cold Drinks are frequently bought together  
- High confidence rules suggest strong product relationships  

## STEP 4: Build Recommendation System

In [49]:
def recommend(product):
    recs = filtered_rules[filtered_rules['antecedents'].str.contains(product)]
    recs = recs.sort_values(by='confidence', ascending=False)
    return recs[['antecedents', 'consequents', 'confidence']].head(10)

    if recommendations.empty:
        return "No recommendations found"

    return recommendations[['antecedents', 'consequents', 'confidence']].sort_values(
        by='confidence', ascending=False
    )

In [50]:
recommend("Milk")

,antecedents,consequents,confidence
252,"Butter, Milk",Cheese,0.450000
212,"Bread, Milk",Cheese,0.437500
155,"Chips, Milk",Biscuits,0.421053
308,"Chips, Milk",Cheese,0.421053
276,"Cold Drink, Milk",Butter,0.411765
59,Milk,Cheese,0.400000
290,"Butter, Milk",Juice,0.400000
291,"Milk, Juice",Butter,0.400000
154,"Biscuits, Milk",Chips,0.380952
198,"Bread, Milk",Butter,0.375000
